In [1]:
import polars as pl
import altair

In [ ]:
"""
Exploration conclusions:
    1. Cols [contract_desc] and [nip] are not atomic - data is paired in a form of comma delimited strings
        
        Example:
        
        [contract_desc]                     [nip]
        "company_a, company_b, company_c" | "6791921738, 8691984858, 78236424843"
        
        Decisions:
        
        There are 14 rows where after splitting data, lengths in columns don't match. These will be removed from further analysis.
        
        Conditions for removal:
        
        - NIP contains comma
        - Different than 0 length of contractor_name and NIP after splitting data in columns by a comma
"""


'\nExploration conclusions:\n    1. Cols [contract_desc] and [nip] are not atomic - data is paired in a form of comma delimited strings\n\n        Example:\n\n        [contract_desc]                     [nip]\n        "company_a, company_b, company_c" | "6791921738, 8691984858, 78236424843"\n\n        Decisions:\n\n        There are 14 rows where after splitting data, lengths in columns don\'t match. These will be removed from further analysis. \n'

In [ ]:
df = pl.read_excel(r"data\umk_contracts_202*.xlsx")

df = df.rename({
    'Lp': 'idx',
    'rok_plik': 'year_filename',
    'Numer w rejestrze': 'registry',
    'Nazwa Kontrahenta ': 'contractor_name',
    'NIP': 'nip',
    'Kwota złotych brutto': 'gross_total_pln',
    'Data obowiązywania od': 'date_valid_from',
    'Data obowiązywania do': 'date_valid_to',
    'Data zawarcia': 'date_contract_signed',
    'Przedmiot umowy': 'contract_desc',
    'Jedn. Realizująca': 'administrative_unit'
})

from types import SimpleNamespace

_names = SimpleNamespace({col: col for col in df.columns})

C:\Users\Manager\AppData\Local\Temp\ipykernel_2012\1031874544.py:1: FutureWarning: from_arrow(<ArrowStreamExportable>) will return a Series instead of a DataFrame in 2.0. To avoid this warning, pass the ArrowStreamExportable to either `pl.DataFrame` or `pl.Series` instead based on your desired output type.
  df = pl.read_excel(r"data\umk_contracts_202*.xlsx")


In [25]:
"""
    1. Cols [contract_desc] and [nip] are not atomic - data is paired in a form of comma delimited strings
        
        Example:
        
        [contract_desc]                     [nip]
        "company_a, company_b, company_c" | "6791921738, 8691984858, 78236424843"
        
        Decisions:
        
        There are 14 rows where after splitting data, lengths in columns don't match. These will be removed from further analysis.
        
        Conditions for removal:
        
        - NIP contains comma
        - Different than 0 length of contractor_name and NIP after splitting data in columns by a comma
"""

mask = (
    df
    .select(
        pl.col(_names.idx, _names.year_filename, _names.contractor_name, _names.nip)
    )
    .with_columns(
        pl.col(_names.nip).str.contains(',').alias("nip_contains_comma")
    )
    .filter(
        pl.col("nip_contains_comma")
    )
    .with_columns(
        pl.col(_names.contractor_name).str.split(','),
        pl.col(_names.nip).str.split(',')
    )
    .with_columns(
        abs(
            pl.col(_names.contractor_name).list.len().cast(pl.Int64)
            - pl.col(_names.nip).list.len().cast(pl.Int64)
            ).alias("diff_len") #cast to Int64 or we get underflow
    )
    .filter(
        pl.col("diff_len") != 0
    )
    .select(
        'idx', 'year_filename'
    )
)

df = df.join(mask, on=['idx', 'year_filename'], how='anti')


In [49]:
(
    df
    .select(
        pl.col(_names.idx, _names.year_filename, _names.contractor_name, _names.nip)
    )
    .with_columns(
        pl.col(_names.nip).str.contains(',').alias("nip_contains_comma")
    )
    .filter(
        ~pl.col("nip_contains_comma")
    )
    .with_columns(
        pl.col(_names.contractor_name).str.split(','),
        pl.col(_names.nip).str.split(',')
    )
    .with_columns(
        abs(
            pl.col(_names.contractor_name).list.len().cast(pl.Int64)
            - pl.col(_names.nip).list.len().cast(pl.Int64)
            ).alias("diff_len") #cast to Int64 or we get underflow
    )
    .filter(
        (pl.col("diff_len") == 1)
        & (pl.col("nip").list.len() == 1)
        & ~(pl.col("contractor_name").list.eval(pl.element().str.contains('S.C')).list.any())
        & ~(pl.col("contractor_name").list.eval(pl.element().str.contains('SPÓŁKA CYWILNA')).list.any())
        & ~(pl.col("contractor_name").list.eval(pl.element().str.contains('SPÓŁKA  PARTNERSKA')).list.any())
        & ~(pl.col("contractor_name").list.eval(pl.element().str.contains('SPOLKA CYWILNA')).list.any())
        
    )
    # .explode("nip")
    # .select(pl.col("nip").str.strip_chars())
    # .unique()
    # .drop_nulls()
)

idx,year_filename,contractor_name,nip,nip_contains_comma,diff_len
i64,i64,list[str],list[str],bool,i64
742,2023,"[""FALL"", "" JAROSŁAW FALL""]","[""6760050254""]",false,1
900,2023,"[""GRZEGORZ SOKOŁOWSKI"", ""GS ENERGIA""]","[""6852211417""]",false,1
911,2023,"[""""JAROSZ"" GEODEZJA "", "" BUDOWNICTWO""]","[""7351035335""]",false,1
1059,2023,"[""MET M. RACZYŃSKI"", "" A. LATACZ SPÓŁKA JAWNA""]","[""899-010-56-46""]",false,1
1098,2023,"[""EWA PERŁAKOWSKA USŁUGI DORADCZE"", "" INFORMATYCZNE I ARCHIWALNE E-PERLA""]","[""6262458660""]",false,1
…,…,…,…,…,…
5341,2025,"[""PRZEDSIĘBIORSTWO HANDLOWOUSŁUGOWE ""KRAKSPORT"" SPÓŁKA JAWNA Z.PIĘTAK"", "" A.PIĘTAK""]","[""6761015389""]",false,1
5348,2025,"[""ALTARO MAGDALENA PIEKARCZYK"", "" MARIUSZ PIEKARCZYK SPÓŁA CYWILNA""]","[""6292474474""]",false,1
5919,2025,"[""AGENCJA REKLAMY EUREKA PLUS BARBARA FEDOROWICZ"", "" RYSZARD FEDOROWICZ""]","[""8131359042""]",false,1


In [44]:
# Filtrujemy tylko te rekordy, z którymi mamy problem, 
# a następnie rozwijamy listy NIP-ów do pojedynczych wartości i bierzemy unikalne.
nipy_do_sprawdzenia = (
    df
    .filter(pl.col("diff_len") != 0)
    .explode("nip_list")
    .select(pl.col("nip_list").str.strip_chars())
    .unique()
    .drop_nulls()
)["nip_list"].to_list()

ColumnNotFoundError: unable to find column "diff_len"; valid columns: ["idx", "year_filename", "registry", "contractor_name", "nip", "gross_total_pln", "date_valid_from", "date_valid_to", "date_contract_signed", "contract_desc", "administrative_unit"]